In [1]:
from pyspark.sql import SparkSession
import os
import json

In [2]:
current_path = os.getcwd()

dir_path = os.path.dirname(os.path.dirname(current_path))

In [3]:
# Obter o diretório atual
current_path = os.getcwd()

# Verificar o ambiente
if "dev" in current_path:
    env = "dev"
elif "prod" in current_path:
    env = "prod"
else:
    env = "unknown"

print(f"A variável de ambiente é: {env}")

A variável de ambiente é: dev


In [4]:
with open(dir_path + f'/{env}-env/projeto_steam/config.json', 'r') as arquivo:
  config = json.load(arquivo)

In [5]:
jar_path = config['jar_path']

# Crie a sessão Spark com o JAR configurado
spark = SparkSession.builder \
    .appName("GoldStep") \
    .config("spark.jars", jar_path) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

24/11/15 19:32:04 WARN Utils: Your hostname, fececa-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
24/11/15 19:32:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


24/11/15 19:32:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [6]:
url = config['url']

properties = {
    "user": config['user'],
    "password": config['password'],
    "driver": config['driver']
}

In [7]:
user = spark.read.jdbc(url=url, table="user", properties=properties)
wish = spark.read.jdbc(url=url, table="wishlist", properties=properties)

In [8]:
union = user.unionByName(wish,allowMissingColumns=True)

In [9]:
union.printSchema()

root
 |-- id: long (nullable = true)
 |-- img: string (nullable = true)
 |-- name: string (nullable = true)
 |-- playtime: double (nullable = true)
 |-- short_description: string (nullable = true)
 |-- website: string (nullable = true)
 |-- recommended: string (nullable = true)
 |-- minimum: string (nullable = true)
 |-- link: string (nullable = true)
 |-- file_date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: long (nullable = true)



In [10]:
union.show()

+-------+--------------------+--------------------+--------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+-----+--------+
|     id|                 img|                name|playtime|   short_description|             website|         recommended|             minimum|                link| file_date|price|discount|
+-------+--------------------+--------------------+--------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+-----+--------+
|    730|https://shared.ak...|    Counter-Strike 2|    1.03|For over two deca...|http://counter-st...|                   -|Minimum:OS: Windo...|https://store.ste...|2024-11-15| NULL|    NULL|
|   4700|https://shared.ak...|Total War: MEDIEV...|   16.65|Spanning the most...|http://www.totalw...|                   -|Minimum System Re...|https://store.ste...|2024-11-15| NULL|    NULL|
|   4760|https://shared.ak...|Rome: Tota

In [11]:
union.write \
.jdbc(url=url, table="steam", mode="overwrite", properties=properties)